# SCP Step 0 - Prepare Tune Directory

This notebook bootstraps a tune directory so it is ready for the refactored
`modules_local` pipeline (Steps 1-6).

What Step 0 can do:
- download/cache Allen bundle files (`manifest.json`, morphology, fit json, `modfiles/`),
- compile modfiles (`nrnivmodl`) and optionally load the DLL,
- scaffold common configs under `cell_configs/`,
- run validation checks (`manifest`, `load_cell`, `inputs.check_inputs`).

Script equivalent:
- `python scripts/step0_prepare.py --cell PV --tune seg_tuned --specimen-id 484635029`


In [1]:
from pathlib import Path
import json
import os
import sys


def _is_scp_root(path: Path) -> bool:
    return (path / 'cells').is_dir() and (path / 'run_pipeline.py').is_file()


def _find_scp_root(start: Path) -> Path:
    env_root = os.environ.get('SCP_ROOT')
    if env_root:
        env_path = Path(env_root).expanduser().resolve()
        if _is_scp_root(env_path):
            return env_path
        raise FileNotFoundError(
            f"SCP_ROOT does not point to an SCP repo root: {env_path}. "
            "Expected cells/ and run_pipeline.py."
        )

    start = start.resolve()
    for cand in (start, *start.parents):
        if _is_scp_root(cand):
            return cand

    for base in (start, start.parent):
        try:
            for child in base.iterdir():
                if child.is_dir() and _is_scp_root(child):
                    return child.resolve()
        except Exception:
            pass

    raise FileNotFoundError(
        f'Could not locate SCP repo root from {start}. '
        'Set SCP_ROOT or launch Jupyter from inside the repo.'
    )


repo_root = _find_scp_root(Path.cwd()).resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('SCP root:', repo_root)


SCP root: /home/hrbncv/SCP


## 0.1 User Settings

Use these toggles to control how Step 0 prepares the tune directory.


In [2]:
# Target location
cell_name = 'SST'               # e.g., PV, SST, SST_0, PN
tune_name = 'adb_peri'        # folder under cells/<cell>/tunes/
tunes_parent = 'tunes'
tune_dir_override = None       # optional absolute/relative path; overrides cell/tune settings

# Allen model identity
specimen_id = 485466109 # SST: 485466109, PV: 484635029 -- set None to use default for known cells
model_type = 'perisomatic'     # e.g., perisomatic, all active

# Cell metadata scaffold (set None to use defaults)
soma_diam_multiplier = 1.0
cell_color = 'magenta'            # for plotting; e.g., 'magenta', 'cyan', 'orange'

# Optional preview
LIST_AVAILABLE_MODELS = False

# Step-0 actions
DO_DOWNLOAD = True
FORCE_DOWNLOAD = False
CACHE_STIMULUS = False

DO_COMPILE_MODFILES = True
RECOMPILE_MODFILES = False
LOAD_COMPILED_DLL = True
SORT_GENOME_ENTRIES_BY_SECTION = False

DO_SCAFFOLD_CONFIGS = True
CONFIG_MODE = 'fill'           # 'fill' | 'overwrite' | 'skip'
SYNC_CELL_METADATA = True

DO_VALIDATE = True
VALIDATE_INPUT_CONFIGS = True


In [3]:
from modules_local import download_cell
from modules_local.step0_prepare import (
    guess_cell_color,
    guess_soma_multiplier,
    guess_specimen_from_cell,
)

if tune_dir_override:
    tune_dir = Path(tune_dir_override).expanduser().resolve()
else:
    tune_dir = (repo_root / 'cells' / cell_name / tunes_parent / tune_name).resolve()

if specimen_id is None:
    specimen_id = guess_specimen_from_cell(cell_name)
if specimen_id is None:
    raise ValueError('specimen_id is required for unknown cell labels; set it explicitly.')

if soma_diam_multiplier is None:
    soma_diam_multiplier = guess_soma_multiplier(cell_name)

if cell_color is None:
    cell_color = guess_cell_color(cell_name)

print('Tune dir:', tune_dir)
print('Cell:', cell_name, '| Tune:', tune_name)
print('Specimen:', specimen_id, '| Model type:', model_type)
print('Soma diam multiplier:', soma_diam_multiplier, '| Color:', cell_color)

if LIST_AVAILABLE_MODELS:
    download_cell.list_ADB_models(specimen_id, filter_type=model_type)


/home/hrbncv/miniconda3/envs/BMTK-py311/lib/python3.11/site-packages/allensdk/model/biophys_sim/config.py:38: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename  # @UnresolvedImport


Tune dir: /home/hrbncv/SCP/cells/SST/tunes/adb_peri
Cell: SST | Tune: adb_peri
Specimen: 485466109 | Model type: perisomatic
Soma diam multiplier: 1.0 | Color: magenta


In [4]:
from modules_local.step0_prepare import prepare_tune

summary = prepare_tune(
    tune_dir=tune_dir,
    cell_name=cell_name,
    tune_name=tune_name,
    specimen_id=int(specimen_id),
    model_type=model_type,
    soma_diam_multiplier=float(soma_diam_multiplier),
    color=cell_color,
    do_download=DO_DOWNLOAD,
    force_download=FORCE_DOWNLOAD,
    cache_stimulus=CACHE_STIMULUS,
    do_compile_modfiles=DO_COMPILE_MODFILES,
    recompile_modfiles=RECOMPILE_MODFILES,
    load_compiled_dll=LOAD_COMPILED_DLL,
    sort_genome_entries_by_section=SORT_GENOME_ENTRIES_BY_SECTION,
    do_scaffold_configs=DO_SCAFFOLD_CONFIGS,
    config_mode=CONFIG_MODE,
    sync_cell_metadata=SYNC_CELL_METADATA,
    do_validate=DO_VALIDATE,
    validate_inputs_cfg=VALIDATE_INPUT_CONFIGS,
)

print(json.dumps(summary, indent=2))


2026-02-14 20:36:16,546 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/496079508
2026-02-14 20:36:16,973 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/496605076
2026-02-14 20:36:17,259 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337293
2026-02-14 20:36:17,501 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337054
2026-02-14 20:36:17,638 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337225
2026-02-14 20:36:17,777 allensdk.api.api.retrieve_file_over_http INFO     Downloading URL: http://api.brain-map.org/api/v2/well_known_file_download/395337019
2026-02-14 20:36:17,918 allensdk.api.api.retrieve_fi

Downloaded model_id=485720616 (Biophysical - perisomatic_Sst-IRES-Cre;Ai14-202729.03.02.01) for specimen_id=485466109
/home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles
Mod files: "./CaDynamics.mod" "./Ca_HVA.mod" "./Ca_LVA.mod" "./Ih.mod" "./Im.mod" "./Im_v2.mod" "./Kd.mod" "./K_P.mod" "./K_T.mod" "./Kv2like.mod" "./Kv3_1.mod" "./Nap.mod" "./NaTa.mod" "./NaTs.mod" "./NaV.mod" "./SK.mod"

Creating 'x86_64' directory for .o files.

 -> Compiling mod_func.cpp
 -> NMODL ../CaDynamics.mod
 -> NMODL ../Ca_HVA.mod
 -> NMODL ../Ca_LVA.mod
 -> NMODL ../Ih.mod
 -> NMODL ../Im.mod
 -> NMODL ../Im_v2.mod
 -> NMODL ../Kd.mod
 -> NMODL ../K_P.mod
 -> NMODL ../K_T.mod
 -> NMODL ../Kv2like.mod
 -> NMODL ../Kv3_1.mod
 -> NMODL ../Nap.mod
 -> NMODL ../NaTa.mod
 -> NMODL ../NaTs.mod
 -> NMODL ../NaV.mod
 -> NMODL ../SK.mod
 -> Compiling CaDynamics.c
 -> Compiling Ca_HVA.c
 -> Compiling Ca_LVA.c
 -> Compiling Ih.c


Translating CaDynamics.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/CaDynamics.c
Translating Ca_HVA.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/Ca_HVA.c
Thread Safe
Translating Ca_LVA.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/Ca_LVA.c
Thread Safe
Thread Safe
Translating Ih.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/Ih.c
Thread Safe
Translating Im.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/Im.c
Translating Im_v2.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/Im_v2.c
Thread Safe
Thread Safe
Translating Kd.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/Kd.c
Thread Safe
Translating K_P.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/K_P.c
Thread Safe
Translating K_T.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles/x86_64/K_T.c
Translating Kv2like.mod into /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles

 -> Compiling Im.c
 -> Compiling Im_v2.c
 -> Compiling Kd.c
 -> Compiling K_P.c
 -> Compiling K_T.c
 -> Compiling Kv2like.c
 -> Compiling Kv3_1.c
 -> Compiling Nap.c
 -> Compiling NaTa.c
 -> Compiling NaTs.c
 -> Compiling NaV.c
 -> Compiling SK.c
 => LINKING shared library ./libnrnmech.so
 => LINKING executable ./special LDFLAGS are:    -pthread
Successfully created x86_64/special
Loaded Allen cell for 'SST' from manifest.json, soma_diam_multiplier=1.0, Vinit=-79.18096923828125, loader=allensdk_default
{
  "tune_dir": "/home/hrbncv/SCP/cells/SST/tunes/adb_peri",
  "cell_name": "SST",
  "tune_name": "adb_peri",
  "specimen_id": 485466109,
  "model_type": "perisomatic",
  "soma_diam_multiplier": 1.0,
  "actions": {
    "download": {
      "status": "ok",
      "model_id": 485720616,
      "model_name": "Biophysical - perisomatic_Sst-IRES-Cre;Ai14-202729.03.02.01",
      "n_files": 21
    },
    "sort_genome": {
      "status": "skipped"
    },
    "compile_modfiles": {
      "status": "o

--No graphics will be displayed.


## 0.2 Quick Check

The paths below should exist after a successful Step 0 run.


In [5]:
check_paths = [
    tune_dir / 'manifest.json',
    tune_dir / 'modfiles',
    tune_dir / 'cell_configs' / 'cell_config.json',
    tune_dir / 'cell_configs' / 'sim_config.json',
    tune_dir / 'cell_configs' / 'geometry.json',
    tune_dir / 'cell_configs' / 'syn_config.json',
    tune_dir / 'cell_configs' / 'syn_groups' / 'placeholder_off.json',
]

for p in check_paths:
    print(f"{'OK' if p.exists() else 'MISSING'}  {p}")


OK  /home/hrbncv/SCP/cells/SST/tunes/adb_peri/manifest.json
OK  /home/hrbncv/SCP/cells/SST/tunes/adb_peri/modfiles
OK  /home/hrbncv/SCP/cells/SST/tunes/adb_peri/cell_configs/cell_config.json
OK  /home/hrbncv/SCP/cells/SST/tunes/adb_peri/cell_configs/sim_config.json
OK  /home/hrbncv/SCP/cells/SST/tunes/adb_peri/cell_configs/geometry.json
OK  /home/hrbncv/SCP/cells/SST/tunes/adb_peri/cell_configs/syn_config.json
OK  /home/hrbncv/SCP/cells/SST/tunes/adb_peri/cell_configs/syn_groups/placeholder_off.json
